# Ariel 2025 — Analysis & Figures cho báo cáo
Phân tích sau train: reliability (calibration), learning curve (GLL vs #planets), bảng kết quả + CV std, predicted-vs-true, RMSE per-wavelength, feature importance, ablation calibration. Tự train model rẻ → self-contained.

In [ ]:
# === Setup: clone running branch and make modules importable ===
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Jun1801/ML_IT3190E_Project.git"
CLONE_DIR = Path("/kaggle/working/ML_IT3190E_Project")
if not CLONE_DIR.exists():
    subprocess.run(["git", "clone", "--branch", "running", "--single-branch", REPO_URL, str(CLONE_DIR)], check=True)
    print("Cloned 'running' →", CLONE_DIR)
else:
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull"], check=True)

for _p in [str(CLONE_DIR / "src"), "/kaggle/input/ariel-ml-src/src", "src", "../src"]:
    if Path(_p).exists():
        sys.path.insert(0, _p); print("Using src from:", _p); break

DATA_ROOT = Path("/kaggle/input/ariel-data-challenge-2025")
OUTPUT_DIR = Path("/kaggle/working"); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = OUTPUT_DIR / "plots"; PLOTS_DIR.mkdir(parents=True, exist_ok=True)
PRECOMPUTED_DIR = CLONE_DIR / "precomputed"
print("DATA_ROOT exists:", DATA_ROOT.exists())


## Load features + targets

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import replace

from config import ModelConfig
from dataset_builder import align_features_and_targets
from training import train_model, cross_validate_model

LIMIT = None; TIME_BINS = 128
N_COMPONENTS = 24; N_SPLITS = 3; SIGMA_CAL_FRACTION = 0.2; RANDOM_STATE = 42

train_csv = PRECOMPUTED_DIR / f"features_train_L{LIMIT}_T{TIME_BINS}.csv"
for cand in [train_csv, OUTPUT_DIR / train_csv.name, Path("/kaggle/input/ariel-features") / train_csv.name]:
    if Path(cand).exists(): train_csv = Path(cand); break
features = pd.read_csv(train_csv)
targets = pd.read_csv(DATA_ROOT / "train.csv")
X, y, groups, target_columns = align_features_and_targets(features, targets)
Xv = X.to_numpy(dtype=float)
PHC = ModelConfig(n_components=N_COMPONENTS, random_state=RANDOM_STATE, sigma_per_target=True)
print("X:", Xv.shape, "| y:", y.shape, "| planets:", len(np.unique(groups)))


## 1. Reliability diagram (calibration)
Coverage thực nghiệm trong khoảng ±zσ vs coverage danh nghĩa. Đường gần chéo = σ calibrate tốt.

In [ ]:
from scipy.stats import norm
res = train_model(Xv, y, model_name="bayesian_ridge", model_config=PHC,
                  validation_fraction=0.25, groups=groups, random_state=RANDOM_STATE)
yv = y[res.validation_index]; mu = res.prediction.mu; sig = res.prediction.sigma
levels = np.linspace(0.1, 0.99, 12)
emp = [float(np.mean(np.abs(yv - mu) <= norm.ppf((1 + p) / 2) * sig)) for p in levels]
plt.figure(figsize=(5.5, 5.5))
plt.plot([0, 1], [0, 1], "k--", lw=1, label="ideal")
plt.plot(levels, emp, "o-", color="crimson", label="bayesian_ridge + PHC")
plt.xlabel("nominal coverage"); plt.ylabel("empirical coverage"); plt.title("Reliability diagram"); plt.legend()
plt.tight_layout(); plt.savefig(PLOTS_DIR / "analysis_reliability.png", dpi=150, bbox_inches="tight"); plt.show()
print("coverage ±1σ:", round(emp[np.argmin(abs(levels-0.6827))] if False else float(np.mean(np.abs(yv-mu)<=sig)),3))


## 2. Learning curve — GLL vs số planet (chứng minh phụ thuộc dữ liệu)

In [ ]:
uniq = np.unique(groups)
sizes = [s for s in [50, 100, 200, 400, 800, len(uniq)] if s <= len(uniq)]
sizes = sorted(set(sizes))
rows = []
for s in sizes:
    keep = np.isin(groups, uniq[:s])
    m = cross_validate_model(Xv[keep], y[keep], model_name="bayesian_ridge", model_config=PHC,
                             n_splits=N_SPLITS, groups=groups[keep], random_state=RANDOM_STATE,
                             sigma_cal_fraction=SIGMA_CAL_FRACTION).mean_metrics
    rows.append({"n_planets": s, "ariel_gll_score": m["ariel_gll_score"], "rmse_mean": m["rmse_mean"]})
lc = pd.DataFrame(rows); lc.to_csv(OUTPUT_DIR / "learning_curve.csv", index=False)
plt.figure(figsize=(7, 4))
plt.plot(lc["n_planets"], lc["ariel_gll_score"], "o-")
plt.xlabel("số planet train"); plt.ylabel("Ariel GLL (CV)"); plt.title("Learning curve")
plt.tight_layout(); plt.savefig(PLOTS_DIR / "analysis_learning_curve.png", dpi=150, bbox_inches="tight"); plt.show()
lc


## 3. Bảng kết quả + CV std (mean ± std qua fold)

In [ ]:
def cv_mean_std(model_name, cfg):
    folds = cross_validate_model(Xv, y, model_name=model_name, model_config=cfg, n_splits=N_SPLITS,
                                 groups=groups, random_state=RANDOM_STATE, sigma_cal_fraction=SIGMA_CAL_FRACTION).fold_results
    def ms(key):
        v = [f.evaluation.as_dict()[key] for f in folds]; return float(np.mean(v)), float(np.std(v))
    g_m, g_s = ms("ariel_gll_score"); r_m, r_s = ms("rmse_mean")
    return {"model": model_name, "GLL_mean": g_m, "GLL_std": g_s, "rmse_mean": r_m, "rmse_std": r_s}

TABLE_MODELS = ["ridge", "bayesian_ridge", "extra_trees"]   # mở rộng tùy ý
tab = pd.DataFrame([cv_mean_std(m, PHC) for m in TABLE_MODELS]).sort_values("GLL_mean", ascending=False)
tab.to_csv(OUTPUT_DIR / "results_with_std.csv", index=False)
print(tab.to_string(index=False))
tab


## 4. Predicted vs true spectrum (+ dải σ)

In [ ]:
pick = res.validation_index[:3]
fig, axes = plt.subplots(1, len(pick), figsize=(5 * len(pick), 4))
for ax, gi, idx in zip(np.atleast_1d(axes), range(len(pick)), range(len(pick))):
    yt = y[res.validation_index][idx]; m = mu[idx]; s = sig[idx]
    xs = np.arange(len(yt))
    ax.plot(xs, yt, color="black", lw=1, label="true")
    ax.plot(xs, m, color="crimson", lw=1, label="pred μ")
    ax.fill_between(xs, m - s, m + s, alpha=0.3, color="crimson", label="±1σ")
    ax.set_xlabel("wavelength index"); ax.legend(fontsize=8)
plt.suptitle("Predicted vs true spectrum"); plt.tight_layout()
plt.savefig(PLOTS_DIR / "analysis_pred_vs_true.png", dpi=150, bbox_inches="tight"); plt.show()


## 5. RMSE per-wavelength

In [ ]:
rmse_wl = np.sqrt(((y[res.validation_index] - mu) ** 2).mean(axis=0))
plt.figure(figsize=(9, 3))
plt.plot(rmse_wl, color="purple"); plt.xlabel("wavelength index"); plt.ylabel("RMSE (val)"); plt.title("RMSE per wavelength")
plt.tight_layout(); plt.savefig(PLOTS_DIR / "analysis_rmse_per_wavelength.png", dpi=150, bbox_inches="tight"); plt.show()


## 6. Feature importance (tree)
Trung bình importance của ExtraTrees qua các PCA component, trọng số = explained variance.

In [ ]:
rf = train_model(Xv, y, model_name="extra_trees", model_config=ModelConfig(n_components=N_COMPONENTS, random_state=RANDOM_STATE),
                 validation_fraction=0.25, groups=groups, random_state=RANDOM_STATE).model
w = rf.pca.explained_variance_ratio_
imp = np.zeros(Xv.shape[1])
for j, est in enumerate(rf.models):
    fi = getattr(est, "feature_importances_", None)
    if fi is not None: imp += w[j] * fi
imp = imp / max(w.sum(), 1e-12)
order = np.argsort(imp)[::-1][:20]
plt.figure(figsize=(8, 6))
plt.barh([X.columns[i] for i in order][::-1], imp[order][::-1], color="seagreen")
plt.xlabel("importance (variance-weighted)"); plt.title("Top-20 feature importance (extra_trees)")
plt.tight_layout(); plt.savefig(PLOTS_DIR / "analysis_feature_importance.png", dpi=150, bbox_inches="tight"); plt.show()


## 7. Ablation calibration: none → scalar → per-wavelength → feature-conditioned

In [ ]:
abl_modes = {
    "none":                ModelConfig(n_components=N_COMPONENTS, random_state=RANDOM_STATE, calibrate_sigma=False),
    "scalar":              ModelConfig(n_components=N_COMPONENTS, random_state=RANDOM_STATE),
    "per_wavelength":      ModelConfig(n_components=N_COMPONENTS, random_state=RANDOM_STATE, sigma_per_target=True),
    "feature_conditioned": ModelConfig(n_components=N_COMPONENTS, random_state=RANDOM_STATE, sigma_feature_conditioned=True),
}
abl = []
for mode, cfg in abl_modes.items():
    m = cross_validate_model(Xv, y, model_name="bayesian_ridge", model_config=cfg, n_splits=N_SPLITS,
                             groups=groups, random_state=RANDOM_STATE, sigma_cal_fraction=SIGMA_CAL_FRACTION).mean_metrics
    abl.append({"calibration": mode, "ariel_gll_score": m["ariel_gll_score"], "gaussian_nll": m["gaussian_nll"]})
abl = pd.DataFrame(abl); abl.to_csv(OUTPUT_DIR / "calibration_ablation.csv", index=False)
plt.figure(figsize=(7, 4))
plt.bar(abl["calibration"], abl["ariel_gll_score"], color="steelblue")
plt.ylabel("Ariel GLL"); plt.title("Ablation: tác dụng của calibration σ"); plt.xticks(rotation=15)
plt.tight_layout(); plt.savefig(PLOTS_DIR / "analysis_calibration_ablation.png", dpi=150, bbox_inches="tight"); plt.show()
abl


## 8. Merge ML + Deep benchmark
Gộp mọi `benchmark*.csv` (ML, có thể từ nhiều session) + `exp5_deep_sequence.csv` (deep) trên cùng metric Ariel GLL → `merged_benchmark.csv` + biểu đồ. Chỉnh `BENCH_DIRS`/`DEEP_CSV` theo nơi attach trên Kaggle.

In [ ]:
import glob

# Nơi chứa kết quả benchmark (đổi theo input attach trên Kaggle)
BENCH_DIRS = [OUTPUT_DIR, Path("/kaggle/input")]
DEEP_CSV = None   # vd: Path("/kaggle/input/ariel-deep/exp5_deep_sequence.csv")

mcols = ["family", "model", "ariel_gll_score", "rmse_mean", "gaussian_nll", "coverage_1sigma", "sigma_mean"]
paths = []
for d in BENCH_DIRS:
    paths += [q for q in glob.glob(str(Path(d) / "**" / "benchmark*.csv"), recursive=True)
              if "merged" not in Path(q).name]
if not paths:
    print("Không thấy benchmark*.csv — attach output các session train_ml rồi chỉnh BENCH_DIRS.")
else:
    ml = pd.concat([pd.read_csv(q) for q in paths], ignore_index=True)
    ml = ml[ml["status"] == "ok"].drop_duplicates(subset="model", keep="first")[mcols].copy()
    ml["type"] = "ml"
    parts = [ml]
    if DEEP_CSV and Path(DEEP_CSV).exists():
        deep = pd.read_csv(DEEP_CSV); deep["family"] = "deep"; deep["type"] = "deep"; deep["sigma_mean"] = np.nan
        parts.append(deep[mcols + ["type"]])
    merged = pd.concat(parts, ignore_index=True).sort_values("ariel_gll_score", ascending=False).reset_index(drop=True)
    merged.to_csv(OUTPUT_DIR / "merged_benchmark.csv", index=False)
    with pd.option_context("display.max_rows", None, "display.width", 200):
        print(merged.to_string(index=False))

    mm = merged.sort_values("ariel_gll_score")
    colors = ["#d9534f" if t == "deep" else "#4f86d9" for t in mm["type"]]
    plt.figure(figsize=(9, max(4, 0.42 * len(mm))))
    plt.barh(mm["model"], mm["ariel_gll_score"], color=colors)
    plt.xlabel("Ariel GLL score (higher = better)"); plt.title("Merged benchmark — ML (blue) vs Deep (red)")
    from matplotlib.patches import Patch
    plt.legend(handles=[Patch(color="#4f86d9", label="ML"), Patch(color="#d9534f", label="Deep")])
    plt.tight_layout(); plt.savefig(PLOTS_DIR / "merged_benchmark.png", dpi=150, bbox_inches="tight"); plt.show()


## Tổng kết
Plot lưu ở `plots/`: reliability, learning_curve, pred_vs_true, rmse_per_wavelength, feature_importance, calibration_ablation. Bảng: `learning_curve.csv`, `results_with_std.csv`, `calibration_ablation.csv`.
- Merge ML+Deep: `merged_benchmark.csv` + `plots/merged_benchmark.png` (bảng + biểu đồ so sánh toàn bộ model trên cùng metric GLL).